# Workshop 2 - Power BI dataset readiness (solution)

Goal: prepare the BI-ready dataset and document Import vs DirectQuery decision.

![Power BI mockup](../assets/images/powerbi_report_mockup.png)

In [ ]:
%run ../setup/00_setup

## Success criteria

You are done when:

1. Summary page source is identified and row count is known.
2. Drill-through/detail source is identified and has date boundaries.
3. Import vs DirectQuery decision is justified.
4. Incremental refresh predicate uses a half-open date window.
5. BI contract and cost guardrails are written.
6. The self-check cell passes.

## Runtime pre-check

Run Modules 2 and 3 before this workshop.

In [ ]:
required_objects = [
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
]

missing = [table for table in required_objects if not spark.catalog.tableExists(table)]
if missing:
    raise Exception("Missing objects. Run Modules 2 and 3 first: " + ", ".join(missing))

print("[OK] Workshop inputs exist")

## Tasks

1. Check if monthly aggregate is enough for the summary page.
2. Check if incremental view has date boundaries.
3. Document BI contract.
4. Decide Import vs live/DirectQuery.
5. List cost guardrails.
6. Validate the expected refresh window predicate.

## Task 1 - summary aggregate

In [ ]:
spark.sql(f"""
SELECT
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month,
  COUNT(*) AS aggregate_rows,
  SUM(revenue) AS revenue
FROM {GOLD}.fact_sales_dashboard_monthly
""").show()

## Task 2 - detail/incremental source

In [ ]:
spark.sql(f"""
SELECT
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date,
  COUNT(*) AS detail_rows
FROM {GOLD}.v_fact_sales_incremental
""").show()

## Task 3 - simulate incremental-refresh window

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS rows_in_window,
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date >= DATE '2025-01-01'
  AND order_date <  DATE '2025-04-01'
""").show()

## Task 4 - BI contract

Suggested BI contract:

| Area | Decision |
|---|---|
| Summary source | `gold.fact_sales_dashboard_monthly` |
| Detail source | `gold.v_fact_sales_incremental` |
| Baseline mode | Import |
| Live option | DirectQuery only for a small operational page on Gold aggregate |
| Refresh | incremental by `order_date`, half-open window |
| Cost guardrail | no visual can query Silver detail |
| Owner | Sales Ops for KPI, BI team for report |

## Suggested decision

Use Import for the executive dashboard baseline. Use DirectQuery/live only for a
small operational page that reads Gold aggregates, not Silver detail.

## Task 5 - self-check

In [ ]:
assert spark.catalog.tableExists(f"{GOLD}.fact_sales_dashboard_monthly"), "Missing monthly aggregate"
assert spark.catalog.tableExists(f"{GOLD}.v_fact_sales_incremental"), "Missing incremental view"

summary_rows = spark.sql(f"SELECT COUNT(*) AS rows FROM {GOLD}.fact_sales_dashboard_monthly").first()["rows"]
detail_rows = spark.sql(f"SELECT COUNT(*) AS rows FROM {GOLD}.v_fact_sales_incremental").first()["rows"]
assert summary_rows > 0, "Summary aggregate is empty"
assert detail_rows > 0, "Incremental view is empty"

window = spark.sql(f"""
SELECT
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date >= DATE '2025-01-01'
  AND order_date <  DATE '2025-04-01'
""").first()
assert str(window["min_date"]) >= "2025-01-01", "Window starts too early"
assert str(window["max_date"]) < "2025-04-01", "Window includes RangeEnd boundary"

print("[OK] Workshop 2 self-check passed")